# Feature Store spines for credit and fraud models

This notebook walks through the Snowflake Feature Store pattern on synthetic card data:
define features **once**, then build a different **spine** for each model and let the
Feature Store join features point-in-time.

| Section | What it shows |
|---|---|
| 1. Setup + synthetic data | Customers, transactions, monthly statements, credit line requests |
| 2. Entity + feature views | One `CUSTOMER` entity, three reusable feature views (two refresh on a schedule) |
| 3. Spines | **Event** spine (fraud), **snapshot** spine (account management), **application** spine (credit decisions) |
| 4. Why point-in-time matters | The same model trained on a naive join vs. a point-in-time join |
| 5. XGBoost | Train exactly as you would today, on a DataFrame from the Feature Store |
| 6. Model Registry + explanations | Log the model, get per-customer reasons for a high-risk score |
| 7. Batch scoring | Score every customer with the same feature definitions and write risk flags to a table |
| 8. Adapting this to your data | What to swap out |

Everything is created in `FEATURE_STORE_DEMO` (change it in the next cell). Runtime: about 5 minutes on a SMALL warehouse.
Run it in a Snowflake Workspace notebook; `snowflake-ml-python`, `xgboost`, and `scikit-learn` are preinstalled.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

DB = "FEATURE_STORE_DEMO"
RAW = f"{DB}.RAW"
FS_SCHEMA = "FEATURE_STORE"
MODEL_SCHEMA = "MODELS"
WH = "FEATURE_STORE_DEMO_WH"

for stmt in [
    f"CREATE DATABASE IF NOT EXISTS {DB}",
    f"CREATE SCHEMA IF NOT EXISTS {RAW}",
    f"CREATE SCHEMA IF NOT EXISTS {DB}.{MODEL_SCHEMA}",
    f"CREATE WAREHOUSE IF NOT EXISTS {WH} WAREHOUSE_SIZE = SMALL AUTO_SUSPEND = 60 INITIALLY_SUSPENDED = TRUE",
    f"USE WAREHOUSE {WH}",
    f"USE SCHEMA {DB}.{MODEL_SCHEMA}",
    "ALTER SESSION SET TIMEZONE = 'UTC'",  # Feature Store reads should run in UTC
]:
    session.sql(stmt).collect()
print("ready")

## 1. Synthetic data

Four source tables stand in for your own:

- `CUSTOMERS`: account open date and credit limit
- `TRANSACTIONS`: card transactions with an `IS_FRAUD` label. A few percent of customers get compromised: small "card testing" charges show up a day or two before larger fraudulent ones.
- `STATEMENTS`: one row per customer per billing cycle with utilization, payment, and days past due (DPD). Some customers hit a financial shock and slide from rising utilization to missed payments to 30+ DPD.
- `CREDIT_LINE_REQUESTS`: credit line increase requests with a 6-month outcome

The generator is deterministic (hash-based), so every run produces the same data. You can skip reading this cell.

In [ ]:
U = lambda *k: f"(ABS(HASH({', '.join(k)})) % 1000000) / 1000000.0"   # deterministic uniform(0,1)

session.sql(f"""
CREATE OR REPLACE TABLE {RAW}.SIM_CUSTOMERS AS
WITH b AS (SELECT SEQ4() + 1 AS N FROM TABLE(GENERATOR(ROWCOUNT => 5000)))
SELECT 'C' || LPAD(N, 5, '0') AS CUSTOMER_ID,
       DATEADD(day, FLOOR({U('N', "'open'")} * 600), '2024-01-01')::DATE AS OPEN_DATE,
       ROUND((1000 + (1 - {U('N', "'risk'")}) * 9000) / 500) * 500 AS CREDIT_LIMIT,
       {U('N', "'risk'")} AS LATENT_RISK,
       IFF({U('N', "'shock'")} < 0.15, LAST_DAY(DATEADD(month, FLOOR({U('N', "'shockm'")} * 20), '2025-03-01')), NULL) AS SHOCK_DATE,
       IFF({U('N', "'comp'")} < 0.04, DATEADD(minute, FLOOR({U('N', "'compts'")} * 60 * 24 * 600), '2025-03-01'::TIMESTAMP_NTZ), NULL) AS COMPROMISE_TS
FROM b
""").collect()

# Public table: the latent risk / shock / compromise columns stay hidden, like they would in real life
session.sql(f"CREATE OR REPLACE TABLE {RAW}.CUSTOMERS AS SELECT CUSTOMER_ID, OPEN_DATE, CREDIT_LIMIT FROM {RAW}.SIM_CUSTOMERS").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {RAW}.STATEMENTS AS
WITH m AS (SELECT LAST_DAY(DATEADD(month, SEQ4(), '2024-01-01'))::DATE AS STATEMENT_DATE
           FROM TABLE(GENERATOR(ROWCOUNT => 34))),
g AS (
  SELECT c.CUSTOMER_ID, m.STATEMENT_DATE, c.CREDIT_LIMIT, c.LATENT_RISK AS R,
         IFF(c.SHOCK_DATE IS NOT NULL AND m.STATEMENT_DATE >= c.SHOCK_DATE,
             LEAST(DATEDIFF(month, c.SHOCK_DATE, m.STATEMENT_DATE) + 1, 5), 0) AS K,
         {U('c.CUSTOMER_ID', 'm.STATEMENT_DATE', "'u'")} AS U1,
         {U('c.CUSTOMER_ID', 'm.STATEMENT_DATE', "'l'")} AS U2,
         {U('c.CUSTOMER_ID', 'm.STATEMENT_DATE', "'s'")} AS U3,
         {U('c.CUSTOMER_ID', 'm.STATEMENT_DATE', "'p'")} AS U4
  FROM {RAW}.SIM_CUSTOMERS c JOIN m ON m.STATEMENT_DATE > c.OPEN_DATE
  WHERE m.STATEMENT_DATE <= '2026-10-31'
),
s AS (
  SELECT *,
         LEAST(1.0, GREATEST(0.0, 0.15 + 0.45 * R + 0.12 * K + (U1 - 0.5) * 0.2)) AS UTILIZATION,
         IFF(U2 < 0.02 + 0.10 * R + 0.15 * K,
             IFF(K = 0, 1 + FLOOR(U3 * (20 + 25 * R)), 5 + FLOOR(U3 * (15 * K + 10))), 0) AS DAYS_PAST_DUE
  FROM g
)
SELECT CUSTOMER_ID, STATEMENT_DATE,
       ROUND(UTILIZATION, 3) AS UTILIZATION,
       ROUND(UTILIZATION * CREDIT_LIMIT, 2) AS BALANCE,
       ROUND(GREATEST(25, UTILIZATION * CREDIT_LIMIT * 0.03), 2) AS MIN_DUE,
       ROUND(IFF(DAYS_PAST_DUE > 0, GREATEST(25, UTILIZATION * CREDIT_LIMIT * 0.03) * U4 * 0.5,
                 GREATEST(25, UTILIZATION * CREDIT_LIMIT * 0.03)
                 + U4 * (1 - R) * GREATEST(0, UTILIZATION * CREDIT_LIMIT * 0.97)), 2) AS PAID_AMOUNT,
       DAYS_PAST_DUE::INT AS DAYS_PAST_DUE
FROM s
""").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {RAW}.TRANSACTIONS AS
WITH normal AS (
  SELECT 'C' || LPAD(FLOOR({U('SEQ4()', "'c'")} * 5000) + 1, 5, '0') AS CUSTOMER_ID, SEQ4() AS I
  FROM TABLE(GENERATOR(ROWCOUNT => 1500000))
),
n2 AS (
  SELECT n.CUSTOMER_ID, n.I, GREATEST(c.OPEN_DATE, '2025-01-01'::DATE) AS START_D,
         {U('n.I', "'d'")} AS UD, {U('n.I', "'h'")} AS UH, {U('n.I', "'a1'")} + {U('n.I', "'a2'")}
         + {U('n.I', "'a3'")} + {U('n.I', "'a4'")} AS UA, {U('n.I', "'m'")} AS UM, {U('n.I', "'p'")} AS UP,
         {U('n.I', "'f'")} AS UF
  FROM normal n JOIN {RAW}.SIM_CUSTOMERS c USING (CUSTOMER_ID)
),
normal_txn AS (
  SELECT CUSTOMER_ID,
         DATEADD(second, FLOOR(IFF(UH < 0.9, 8 + UH / 0.9 * 14, UH * 24) * 3600),
                 DATEADD(day, FLOOR(UD * DATEDIFF(day, START_D, '2026-11-24')), START_D)::TIMESTAMP_NTZ) AS TXN_TS,
         ROUND(EXP(3.0 + 0.9 * (UA - 2) * 1.7), 2)::NUMBER(12, 2) AS AMOUNT,
         DECODE(FLOOR(UM * 8), 0, 'GROCERY', 1, 'GROCERY', 2, 'DINING', 3, 'DINING', 4, 'FUEL',
                5, 'ONLINE_RETAIL', 6, 'TRAVEL', 'ELECTRONICS') AS MERCHANT_CATEGORY,
         UP < 0.7 AS CARD_PRESENT,
         UF < 0.0005 AS IS_FRAUD
  FROM n2
),
comp AS (
  SELECT c.CUSTOMER_ID, c.COMPROMISE_TS, g.SEQ AS J
  FROM {RAW}.SIM_CUSTOMERS c,
       (SELECT SEQ4() AS SEQ FROM TABLE(GENERATOR(ROWCOUNT => 7))) g
  WHERE c.COMPROMISE_TS IS NOT NULL AND c.COMPROMISE_TS > DATEADD(day, 30, c.OPEN_DATE)
),
attack_txn AS (
  SELECT CUSTOMER_ID,
         IFF(J < 4, DATEADD(minute, -FLOOR((6 + {U('CUSTOMER_ID', 'J', "'t'")} * 40) * 60), COMPROMISE_TS),
                    DATEADD(minute, FLOOR({U('CUSTOMER_ID', 'J', "'t'")} * 20 * 60), COMPROMISE_TS)) AS TXN_TS,
         ROUND(IFF(J < 4, 1 + {U('CUSTOMER_ID', 'J', "'a'")} * 3, 200 + {U('CUSTOMER_ID', 'J', "'a'")} * 1300), 2)::NUMBER(12, 2) AS AMOUNT,
         IFF(J < 4, 'ONLINE_RETAIL', DECODE(J, 4, 'ELECTRONICS', 5, 'TRAVEL', 'ONLINE_RETAIL')) AS MERCHANT_CATEGORY,
         FALSE AS CARD_PRESENT,
         J >= 4 AS IS_FRAUD            -- card-testing charges (J < 4) are not labelled fraud
  FROM comp
)
SELECT 'T' || LPAD(ROW_NUMBER() OVER (ORDER BY TXN_TS, CUSTOMER_ID), 8, '0') AS TXN_ID, *
FROM (SELECT * FROM normal_txn UNION ALL SELECT * FROM attack_txn)
WHERE TXN_TS < '2026-11-24'
""").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {RAW}.CREDIT_LINE_REQUESTS AS
WITH r AS (
  SELECT c.CUSTOMER_ID,
         DATEADD(minute, FLOOR({U('c.CUSTOMER_ID', "'rq'")} * 60 * 24 * 300), '2025-06-01'::TIMESTAMP_NTZ) AS REQUEST_TS,
         ROUND(c.CREDIT_LIMIT * (1.2 + {U('c.CUSTOMER_ID', "'amt'")} * 0.8) / 500) * 500 AS REQUESTED_LIMIT
  FROM {RAW}.SIM_CUSTOMERS c
  WHERE {U('c.CUSTOMER_ID', "'req'")} < 0.30
    AND DATEADD(minute, FLOOR({U('c.CUSTOMER_ID', "'rq'")} * 60 * 24 * 300), '2025-06-01'::TIMESTAMP_NTZ)
        > DATEADD(day, 90, c.OPEN_DATE)
)
SELECT 'R' || LPAD(ROW_NUMBER() OVER (ORDER BY r.REQUEST_TS), 6, '0') AS REQUEST_ID,
       r.CUSTOMER_ID, r.REQUEST_TS, r.REQUESTED_LIMIT,
       MAX(IFF(s.DAYS_PAST_DUE >= 30, 1, 0)) AS DEFAULTED_6M
FROM r LEFT JOIN {RAW}.STATEMENTS s
  ON s.CUSTOMER_ID = r.CUSTOMER_ID
 AND s.STATEMENT_DATE > r.REQUEST_TS AND s.STATEMENT_DATE <= DATEADD(month, 6, r.REQUEST_TS)
GROUP BY r.CUSTOMER_ID, r.REQUEST_TS, r.REQUESTED_LIMIT
""").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {RAW}.CALENDAR AS
SELECT DATEADD(day, SEQ4(), '2025-01-01')::DATE AS D
FROM TABLE(GENERATOR(ROWCOUNT => 692))
""").collect()
print("synthetic data created")

In [ ]:
%%sql -r table_counts
SELECT 'CUSTOMERS' AS TABLE_NAME, COUNT(*) AS ROWS_, NULL AS POSITIVE_LABELS FROM {{RAW}}.CUSTOMERS
UNION ALL SELECT 'TRANSACTIONS', COUNT(*), COUNT_IF(IS_FRAUD) FROM {{RAW}}.TRANSACTIONS
UNION ALL SELECT 'STATEMENTS', COUNT(*), COUNT_IF(DAYS_PAST_DUE >= 30) FROM {{RAW}}.STATEMENTS
UNION ALL SELECT 'CREDIT_LINE_REQUESTS', COUNT(*), SUM(DEFAULTED_6M) FROM {{RAW}}.CREDIT_LINE_REQUESTS

## 2. Entity and feature views

A **feature view** is a named, versioned feature definition: a SQL or Snowpark query keyed on an
entity, with a timestamp that says *when each value became known*. Defining features here instead
of inside a training notebook is what makes them reusable: the fraud model, the account
management model, and the credit-decision model below all draw on the same three views.

| Feature view | Grain | Timestamp means | Refresh |
|---|---|---|---|
| `TXN_BEHAVIOR` | customer x day | start of the next day (a day's spend is only complete then) | Managed, daily |
| `PAYMENT_BEHAVIOR` | customer x billing cycle | statement date | Managed, daily |
| `ACCOUNT_PROFILE` | customer | account open date | External view, always current |

A **managed** feature view (one with `refresh_freq`) is backed by a dynamic table that Snowflake
keeps up to date on that schedule, from daily down to near-continuous. Nothing in your pipeline
has to re-run to refresh it. An **external** feature view (no `refresh_freq`) is just a view, for
data that some other process already maintains.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode

fs = FeatureStore(
    session=session,
    database=DB,
    name=FS_SCHEMA,
    default_warehouse=WH,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

customer = Entity(name="CUSTOMER", join_keys=["CUSTOMER_ID"], desc="A cardholder account")
fs.register_entity(customer)
fs.list_entities().show()

In [ ]:
txn_behavior_sql = f"""
WITH daily AS (
    SELECT CUSTOMER_ID, TO_DATE(TXN_TS) AS D,
           COUNT(*) AS N, SUM(AMOUNT) AS AMT,
           COUNT_IF(AMOUNT < 5) AS SMALL_N, COUNT_IF(NOT CARD_PRESENT) AS CNP_N
    FROM {RAW}.TRANSACTIONS GROUP BY 1, 2
),
grid AS (   -- one row per customer per day, so rolling windows count quiet days too
    SELECT c.CUSTOMER_ID, cal.D
    FROM {RAW}.CUSTOMERS c JOIN {RAW}.CALENDAR cal ON cal.D >= GREATEST(c.OPEN_DATE, '2025-01-01'::DATE)
),
x AS (
    SELECT g.CUSTOMER_ID, g.D, COALESCE(N, 0) N, COALESCE(AMT, 0) AMT,
           COALESCE(SMALL_N, 0) SMALL_N, COALESCE(CNP_N, 0) CNP_N
    FROM grid g LEFT JOIN daily d ON d.CUSTOMER_ID = g.CUSTOMER_ID AND d.D = g.D
)
SELECT CUSTOMER_ID,
       DATEADD(day, 1, D)::TIMESTAMP_NTZ AS FEATURE_TS,
       SUM(N)       OVER (PARTITION BY CUSTOMER_ID ORDER BY D ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)  AS TXN_COUNT_7D,
       SUM(N)       OVER (PARTITION BY CUSTOMER_ID ORDER BY D ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS TXN_COUNT_30D,
       SUM(AMT)     OVER (PARTITION BY CUSTOMER_ID ORDER BY D ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)  AS SPEND_7D,
       SUM(AMT)     OVER (PARTITION BY CUSTOMER_ID ORDER BY D ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS SPEND_30D,
       SPEND_30D / NULLIF(TXN_COUNT_30D, 0)                                                              AS AVG_TXN_AMOUNT_30D,
       SUM(SMALL_N) OVER (PARTITION BY CUSTOMER_ID ORDER BY D ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)  AS SMALL_TXN_COUNT_3D,
       SUM(CNP_N)   OVER (PARTITION BY CUSTOMER_ID ORDER BY D ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)  AS CNP_TXN_COUNT_3D
FROM x
"""

payment_behavior_sql = f"""
SELECT CUSTOMER_ID,
       STATEMENT_DATE::TIMESTAMP_NTZ AS FEATURE_TS,
       UTILIZATION,
       UTILIZATION - LAG(UTILIZATION, 3) OVER (PARTITION BY CUSTOMER_ID ORDER BY STATEMENT_DATE) AS UTILIZATION_CHANGE_3M,
       DAYS_PAST_DUE,
       MAX(DAYS_PAST_DUE) OVER (PARTITION BY CUSTOMER_ID ORDER BY STATEMENT_DATE ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS MAX_DPD_3M,
       SUM(IFF(DAYS_PAST_DUE > 0, 1, 0)) OVER (PARTITION BY CUSTOMER_ID ORDER BY STATEMENT_DATE ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS LATE_CYCLES_6M,
       PAID_AMOUNT / NULLIF(MIN_DUE, 0) AS PAYMENT_TO_MIN_RATIO
FROM {RAW}.STATEMENTS
"""

account_profile_sql = f"""
SELECT CUSTOMER_ID, OPEN_DATE::TIMESTAMP_NTZ AS FEATURE_TS, CREDIT_LIMIT
FROM {RAW}.CUSTOMERS
"""

txn_fv = fs.register_feature_view(
    FeatureView(name="TXN_BEHAVIOR", entities=[customer], feature_df=session.sql(txn_behavior_sql),
                timestamp_col="FEATURE_TS", refresh_freq="1 day",
                desc="Rolling spend and velocity features, as of the start of each day"),
    version="V1", overwrite=True)

payment_fv = fs.register_feature_view(
    FeatureView(name="PAYMENT_BEHAVIOR", entities=[customer], feature_df=session.sql(payment_behavior_sql),
                timestamp_col="FEATURE_TS", refresh_freq="1 day",
                desc="Utilization and delinquency history, as of each statement date"),
    version="V1", overwrite=True)

profile_fv = fs.register_feature_view(
    FeatureView(name="ACCOUNT_PROFILE", entities=[customer], feature_df=session.sql(account_profile_sql),
                timestamp_col="FEATURE_TS",   # no refresh_freq -> external feature view (a plain view)
                desc="Static account attributes"),
    version="V1", overwrite=True)

fs.list_feature_views().select("NAME", "VERSION", "REFRESH_FREQ", "DESC").show()

## 3. Spines

The **spine** is the one model-specific piece. It answers three questions: *which entity*, *at what
moment was the decision made*, and *what happened afterwards (the label)*. The Feature Store then
attaches each feature **as it was known at that moment**, an as-of join, so nothing from after the
decision leaks in.

Different decisions need differently shaped spines over the same features:

| Spine | One row per | Timestamp | Label | Model |
|---|---|---|---|---|
| **Event** | transaction | transaction time | was it fraud? | fraud |
| **Snapshot** | account x billing cycle | statement date | 30+ DPD within the next two cycles? | account management (limit cuts, collections) |
| **Application** | credit line request | request time | 30+ DPD within 6 months? | credit decisioning |

A spine can also carry request-time context that isn't a stored feature (the transaction amount,
hour, and card-present flag below). Those pass straight through to the training set.

### 3a. Event spine: fraud

Fraud is rare, so the spine keeps every fraud transaction and a 5% sample of legitimate ones.
Down-sampling inside the spine is a common way to keep training sets manageable.

In [ ]:
fraud_spine = session.sql(f"""
    SELECT TXN_ID, CUSTOMER_ID, TXN_TS, AMOUNT,
           HOUR(TXN_TS) AS TXN_HOUR, IFF(CARD_PRESENT, 1, 0) AS CARD_PRESENT,
           IFF(IS_FRAUD, 1, 0) AS IS_FRAUD
    FROM {RAW}.TRANSACTIONS
    WHERE TXN_TS >= '2025-02-01'
      AND (IS_FRAUD OR ABS(HASH(TXN_ID)) % 20 = 0)
""")

fraud_train = fs.generate_training_set(
    spine_df=fraud_spine,
    features=[txn_fv, profile_fv],
    spine_timestamp_col="TXN_TS",
    spine_label_cols=["IS_FRAUD"],
)
fraud_train.limit(5).show()

### 3b. Snapshot spine: account management

One row per account per billing cycle, for accounts that are not already 30+ DPD. The label looks
*forward* two cycles. This is the shape you'd use for credit limit reductions, early collections
outreach, or a monthly risk refresh of the whole book.

In [ ]:
acct_spine = session.sql(f"""
    WITH s AS (
        SELECT CUSTOMER_ID, STATEMENT_DATE, DAYS_PAST_DUE,
               MAX(IFF(DAYS_PAST_DUE >= 30, 1, 0)) OVER (
                   PARTITION BY CUSTOMER_ID ORDER BY STATEMENT_DATE
                   ROWS BETWEEN 1 FOLLOWING AND 2 FOLLOWING) AS WILL_HIT_30DPD,
               COUNT(*) OVER (PARTITION BY CUSTOMER_ID ORDER BY STATEMENT_DATE
                   ROWS BETWEEN 1 FOLLOWING AND 2 FOLLOWING) AS FUTURE_CYCLES
        FROM {RAW}.STATEMENTS
    )
    SELECT CUSTOMER_ID, STATEMENT_DATE::TIMESTAMP_NTZ AS SNAPSHOT_TS, WILL_HIT_30DPD
    FROM s
    WHERE DAYS_PAST_DUE < 30 AND FUTURE_CYCLES = 2 AND STATEMENT_DATE >= '2025-03-31'
""")

acct_train = fs.generate_training_set(
    spine_df=acct_spine,
    features=[payment_fv, txn_fv, profile_fv],
    spine_timestamp_col="SNAPSHOT_TS",
    spine_label_cols=["WILL_HIT_30DPD"],
)
acct_train.limit(5).show()

### 3c. Application spine: credit line increase requests

Same feature views again, just joined as of the moment each request came in. No new feature
code was needed for a third model.

In [ ]:
app_spine = session.table(f"{RAW}.CREDIT_LINE_REQUESTS").select(
    "REQUEST_ID", "CUSTOMER_ID", "REQUEST_TS", "REQUESTED_LIMIT", "DEFAULTED_6M")

app_train = fs.generate_training_set(
    spine_df=app_spine,
    features=[payment_fv, txn_fv, profile_fv],
    spine_timestamp_col="REQUEST_TS",
    spine_label_cols=["DEFAULTED_6M"],
)
app_train.limit(5).show()

## 4. Why point-in-time matters

A common shortcut is to join each customer's *current* features onto historical labels. The model
then learns from feature values that were only known after each decision, and that doesn't hold up
when it runs in production.

Below, the same XGBoost model is trained twice on the same snapshot spine and the same
out-of-time test window (cycles from March 2026 onward):

- **Naive**: each customer's latest payment features, whenever they were computed
- **Point-in-time**: the training set from 3b

The naive model is then scored the way production would score it, on features as of each
decision date. Its offline number doesn't survive that; the point-in-time model's does.

In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from snowflake.snowpark import Window, functions as F

ACCT_FEATURES = ["UTILIZATION", "UTILIZATION_CHANGE_3M", "DAYS_PAST_DUE", "MAX_DPD_3M", "LATE_CYCLES_6M",
                 "PAYMENT_TO_MIN_RATIO", "TXN_COUNT_30D", "SPEND_30D", "AVG_TXN_AMOUNT_30D", "CREDIT_LIMIT"]
SPLIT = pd.Timestamp("2026-03-01")

def fit_and_score(df, label):
    df = df.copy()
    train, test = df[df["SNAPSHOT_TS"] < SPLIT], df[df["SNAPSHOT_TS"] >= SPLIT]
    m = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, eval_metric="auc")
    m.fit(train[ACCT_FEATURES], train[label])
    return m, train, test, roc_auc_score(test[label], m.predict_proba(test[ACCT_FEATURES])[:, 1])

# Naive: latest payment row per customer, joined onto every historical snapshot
latest_payment = (payment_fv.feature_df
    .with_column("RN", F.row_number().over(Window.partition_by("CUSTOMER_ID").order_by(F.col("FEATURE_TS").desc())))
    .filter(F.col("RN") == 1).drop("RN", "FEATURE_TS"))
naive_df = (acct_spine.join(latest_payment, on="CUSTOMER_ID")
            .join(fs.retrieve_feature_values(acct_spine.select("CUSTOMER_ID", "SNAPSHOT_TS"),
                                             features=[txn_fv, profile_fv],
                                             spine_timestamp_col="SNAPSHOT_TS"),
                  on=["CUSTOMER_ID", "SNAPSHOT_TS"])
            .to_pandas())

pit_df = acct_train.to_pandas()

naive_model, _, _, naive_auc = fit_and_score(naive_df, "WILL_HIT_30DPD")
acct_model, acct_X_train, acct_test, pit_auc = fit_and_score(pit_df, "WILL_HIT_30DPD")

# In production the naive model only ever sees features as of the decision date
naive_in_prod_auc = roc_auc_score(acct_test["WILL_HIT_30DPD"], naive_model.predict_proba(acct_test[ACCT_FEATURES])[:, 1])

print(f"Naive model, offline test AUC:          {naive_auc:.3f}")
print(f"Naive model, scored on real-time features: {naive_in_prod_auc:.3f}   <- what it actually delivers in production")
print(f"Point-in-time model, test AUC:          {pit_auc:.3f}   <- offline and production now agree")

## 5. Train XGBoost (fraud)

Nothing about model training changes. `generate_training_set` hands back a Snowpark DataFrame,
`.to_pandas()` turns it into the pandas DataFrame you'd already pass to XGBoost.

`AMOUNT_VS_AVG_30D` is computed here, not in a feature view: it mixes a request-time value
(this transaction's amount) with a stored feature. Model-specific transforms like this belong
in the model code.

In [ ]:
FRAUD_FEATURES = ["AMOUNT", "TXN_HOUR", "CARD_PRESENT", "AMOUNT_VS_AVG_30D", "TXN_COUNT_7D",
                  "TXN_COUNT_30D", "SPEND_7D", "SMALL_TXN_COUNT_3D", "CNP_TXN_COUNT_3D", "CREDIT_LIMIT"]

fraud_df = fraud_train.to_pandas()
fraud_df["AMOUNT_VS_AVG_30D"] = fraud_df["AMOUNT"] / fraud_df["AVG_TXN_AMOUNT_30D"]

tr, te = fraud_df[fraud_df["TXN_TS"] < SPLIT], fraud_df[fraud_df["TXN_TS"] >= SPLIT]
fraud_model = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, eval_metric="aucpr",
                            scale_pos_weight=(tr["IS_FRAUD"] == 0).sum() / max((tr["IS_FRAUD"] == 1).sum(), 1))
fraud_model.fit(tr[FRAUD_FEATURES], tr["IS_FRAUD"])
print(f"Fraud model test AUC: {roc_auc_score(te['IS_FRAUD'], fraud_model.predict_proba(te[FRAUD_FEATURES])[:, 1]):.3f}")
print(f"Train rows: {len(tr):,}  fraud: {int(tr['IS_FRAUD'].sum())}   Test rows: {len(te):,}  fraud: {int(te['IS_FRAUD'].sum())}")

pd.Series(fraud_model.feature_importances_, index=FRAUD_FEATURES).sort_values(ascending=False)

## 6. Model Registry and per-decision explanations

Logging the account management model to the Model Registry versions it next to the feature views
that trained it (lineage), and enables an `explain` method that returns Shapley values for each
prediction. For a credit limit cut, that's the per-customer list of what drove the score.

The `sample_input_data` rows are the background data the explanations compare against.
Explainability is currently a preview feature.

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name=DB, schema_name=MODEL_SCHEMA)
models = reg.show_models()
if not models.empty and "ACCOUNT_RISK_30DPD" in models["name"].values:   # so the notebook can be re-run
    reg.delete_model("ACCOUNT_RISK_30DPD")

acct_mv = reg.log_model(
    acct_model,
    model_name="ACCOUNT_RISK_30DPD",
    version_name="V1",
    sample_input_data=acct_X_train[ACCT_FEATURES].sample(500, random_state=7),
    target_platforms=["WAREHOUSE"],
    options={"enable_explainability": True},
    comment="30+ DPD within two cycles; trained on FEATURE_STORE spines (PAYMENT_BEHAVIOR, TXN_BEHAVIOR, ACCOUNT_PROFILE)",
)
acct_mv.show_functions()

In [ ]:
# Explain the highest-risk account in the test window
test_scores = acct_model.predict_proba(acct_test[ACCT_FEATURES])[:, 1]
top = acct_test.assign(RISK_SCORE=test_scores).sort_values("RISK_SCORE", ascending=False).head(1)
print(f"Customer {top['CUSTOMER_ID'].iloc[0]} on {top['SNAPSHOT_TS'].iloc[0]:%Y-%m-%d}: risk score {top['RISK_SCORE'].iloc[0]:.2f}")

expl = acct_mv.run(top[ACCT_FEATURES], function_name="explain")
contrib = pd.DataFrame({
    "FEATURE": ACCT_FEATURES,
    "VALUE": top[ACCT_FEATURES].iloc[0].values,
    "CONTRIBUTION": [expl[f"{f}_explanation"].iloc[0] for f in ACCT_FEATURES],
}).sort_values("CONTRIBUTION", ascending=False)
contrib   # positive = pushed toward high risk; these become reason codes

## 7. Batch scoring with the same features

Scoring retrieves the **same** feature views as of *now* and writes a flag table your analytics
teams can query. This replaces running the model locally and sharing the output.

In [ ]:
score_spine = session.table(f"{RAW}.CUSTOMERS").select(
    "CUSTOMER_ID", F.current_timestamp().cast("TIMESTAMP_NTZ").alias("SCORE_TS"))

current_features = fs.retrieve_feature_values(
    spine_df=score_spine, features=[payment_fv, txn_fv, profile_fv], spine_timestamp_col="SCORE_TS")

scored = acct_mv.run(current_features, function_name="predict_proba")
prob_col = [c for c in scored.columns if c.strip('"').endswith("_1")][0]

flags = scored.select(
    "CUSTOMER_ID", "SCORE_TS",
    F.col(prob_col).alias("RISK_SCORE"),
    F.iff(F.col(prob_col) >= 0.5, F.lit("HIGH"), F.lit("LOW")).alias("RISK_FLAG"),
    F.lit("ACCOUNT_RISK_30DPD/V1").alias("MODEL_VERSION"))
flags.write.save_as_table(f"{DB}.{MODEL_SCHEMA}.ACCOUNT_RISK_FLAGS", mode="overwrite")
print("written", f"{DB}.{MODEL_SCHEMA}.ACCOUNT_RISK_FLAGS")

In [ ]:
%%sql -r risk_flag_summary
SELECT RISK_FLAG, COUNT(*) AS CUSTOMERS, ROUND(AVG(RISK_SCORE), 3) AS AVG_SCORE
FROM {{DB}}.{{MODEL_SCHEMA}}.ACCOUNT_RISK_FLAGS
GROUP BY RISK_FLAG ORDER BY RISK_FLAG

To run this on a schedule, schedule this notebook (or just the cells in sections 2 and 7) from
the notebook's **Schedule** menu, or move the scoring cell into a Snowflake task. The feature views
refresh themselves daily, so the scoring job only has to read them.

## 8. Adapting this to your data

| In this notebook | Replace with |
|---|---|
| `RAW.*` synthetic tables | Your transactions, statements, and accounts tables |
| The three feature view queries | Your existing feature SQL. One feature view per grain/source, timestamped by when the value becomes known |
| `fraud_spine` / `acct_spine` / `app_spine` | The decision events and labels from your current training data |
| `XGBClassifier(...)` | Your existing XGBoost code, unchanged. Only the input DataFrame comes from `generate_training_set` |
| `ACCOUNT_RISK_FLAGS` | Wherever downstream teams already read scores |

Rules of thumb:
- Set each feature view's timestamp to when the value was **known**, not when the event happened (e.g. a day's spend is known at the start of the next day).
- Always pass `spine_timestamp_col`. Without it you get latest values, which is the naive join from section 4.
- Keep model-specific transforms (ratios against request values, encoding, scaling) in model code. Keep reusable aggregates in feature views.

**Cleanup:** `DROP DATABASE FEATURE_STORE_DEMO; DROP WAREHOUSE FEATURE_STORE_DEMO_WH;`

Docs: [Feature Store](https://docs.snowflake.com/en/developer-guide/snowflake-ml/feature-store/overview) ·
[Model explainability](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/model-explainability)